<a href="https://colab.research.google.com/github/braadly/EEL4810_Project1/blob/main/brain_tumor_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset is located within shared google drive. Sign in to access.

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import os
print(os.path.exists('/content/drive/MyDrive/train'))
print(os.listdir('/content/drive/MyDrive/train'))

True
['no_tumor', 'Glioblastoma', 'meningioma_tumor', 'pituitary_tumor', 'glioma_tumor', 'Schwannoma', 'Metastatic']


In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
import numpy as np
import torch.nn.init as init
import time

torch.manual_seed(73)
np.random.seed(73)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- load dataset ---------- #
# location of the drive in my personal google drive -Bradly
# the way i set this up, is after Ethan shared the drive with me
# right click the folder, organize, add shortcut, then add it onto
# 'myDrive'

# loaded


DATASET_PATH = '/content/drive/MyDrive/train'

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = ImageFolder(root=DATASET_PATH, transform=train_transforms)

print(f"Total images:  {len(full_dataset)}")
print(f"Classes found: {full_dataset.classes}")
print(f"Class mapping: {full_dataset.class_to_idx}")


# ---------- set up resnet18 neural network ---------- #
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) # sets up resnet18 pretrained model

model.fc = nn.Linear(model.fc.in_features, 7) # creates output layer

init.kaiming_normal_(model.fc.weight, mode='fan_in', nonlinearity='relu')
init.zeros_(model.fc.bias) # initialises output layer weights (randoms for ReLu) and biases (zeros)


# ---------- fine tune model using LoRA rank-4 ---------- # (no need to set up until after midterm report)


# ---------- Training loop ---------- #


# implement method to save and load trained networks? (depending on how long training takes, we may not need this)


# ---------- test with Macro-F1 value and graphs ---------- #



Using device: cuda
Total images:  6007
Classes found: ['Glioblastoma', 'Metastatic', 'Schwannoma', 'glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Class mapping: {'Glioblastoma': 0, 'Metastatic': 1, 'Schwannoma': 2, 'glioma_tumor': 3, 'meningioma_tumor': 4, 'no_tumor': 5, 'pituitary_tumor': 6}


Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [25]:
from sklearn.metrics import f1_score

# train/val split
indices = list(range(len(full_dataset)))
train_indices, val_indices = train_test_split(indices, test_size=0.2, random_state=73)

train_loader = DataLoader(Subset(full_dataset, train_indices), batch_size=32, shuffle=True)
val_loader   = DataLoader(Subset(full_dataset, val_indices),   batch_size=32, shuffle=False)

# optimizer and loss with class weights
model = model.to(device)

labels_list = [label for _, label in full_dataset.samples]
class_counts = Counter(labels_list)
total_samples = sum(class_counts.values())
weights = [total_samples / class_counts[i] for i in range(7)]
weights = torch.tensor(weights, dtype=torch.float).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0002)
criterion = nn.CrossEntropyLoss(weight=weights)

NUM_EPOCHS = 20
best_f1 = 0

for epoch in range(NUM_EPOCHS):
    # train
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # validate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Macro-F1: {macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model.pth')
        print(f"  New best model saved! F1: {best_f1:.4f}")

Epoch 1/20 | Loss: 0.3964 | Macro-F1: 0.9119
  New best model saved! F1: 0.9119
Epoch 2/20 | Loss: 0.1560 | Macro-F1: 0.9269
  New best model saved! F1: 0.9269
Epoch 3/20 | Loss: 0.1196 | Macro-F1: 0.9361
  New best model saved! F1: 0.9361
Epoch 4/20 | Loss: 0.0924 | Macro-F1: 0.9662
  New best model saved! F1: 0.9662
Epoch 5/20 | Loss: 0.0708 | Macro-F1: 0.9304
Epoch 6/20 | Loss: 0.0551 | Macro-F1: 0.9679
  New best model saved! F1: 0.9679
Epoch 7/20 | Loss: 0.0390 | Macro-F1: 0.9662
Epoch 8/20 | Loss: 0.0670 | Macro-F1: 0.9547
Epoch 9/20 | Loss: 0.0522 | Macro-F1: 0.9560
Epoch 10/20 | Loss: 0.0513 | Macro-F1: 0.9441
Epoch 11/20 | Loss: 0.0266 | Macro-F1: 0.9832
  New best model saved! F1: 0.9832
Epoch 12/20 | Loss: 0.0114 | Macro-F1: 0.9719
Epoch 13/20 | Loss: 0.0154 | Macro-F1: 0.9625
Epoch 14/20 | Loss: 0.0306 | Macro-F1: 0.9574
Epoch 15/20 | Loss: 0.0378 | Macro-F1: 0.9740
Epoch 16/20 | Loss: 0.0236 | Macro-F1: 0.9763
Epoch 17/20 | Loss: 0.0564 | Macro-F1: 0.9559
Epoch 18/20 | Los